## Setup

In [1]:
import kagglehub
import os


path = kagglehub.dataset_download("yorkyong/text8-zip")

text8_file_path = os.path.join(path, 'text8')

with open(text8_file_path, 'r') as f:
    text_content = f.read()

words = text_content.split()

print(f'Successfully read text8 file. Total words: {len(words)}')
print(f'First 10 words: {words[:10]}')


100%|██████████| 32.1M/32.1M [00:00<00:00, 92.1MB/s]

Extracting files...


Successfully read text8 file. Total words: 17005207
First 10 words: ['anarchism', 'originated', 'as', 'a', 'term', 'of', 'abuse', 'first', 'used', 'against']


In [16]:
words = words[:50000]

In [27]:
# building the vocabulary
from collections import Counter
wordApparitions = Counter(words)
V = len(wordApparitions)
word2idx = {word: idx for idx, word in enumerate(wordApparitions)}

idx2word = {idx: word for word, idx in word2idx.items()}


In [28]:
import numpy as np
# noise distribution
values = np.array(list(wordApparitions.values()))**(3/4)
total = np.sum(values)
noise_Distribution = values/total

In [29]:
#hyper param initialization
epochs = 5
N =  100
learning_rate = 0.025
windowSize = 5
K = 6 # nr of negative samples
min_lr = learning_rate * 0.0001
W = np.random.randn(V, N) * 0.01
W_prim = np.random.randn(V, N) * 0.01


## Training loop

In [30]:
def sigmoid_scalar(x):
    if x >= 0:
        out =  1. / (1. + np.exp(-x))
    else:
        out = np.exp(x) / (1. + np.exp(x))
    return max(1e-15, min(1 - 1e-15, out))

def sigmoid_vector(x):
    x = np.asarray(x)
    out = 1. / (1. + np.exp(-np.clip(x, -15, 15)))
    return 1. / (1. + np.exp(-np.clip(x, -500, 500)))

In [32]:
for epoch in range(epochs):
  total_loss = 0

  for (i, center_word) in enumerate(words):
    context_words = []

    for j in range(max(0, i-windowSize), min(i+windowSize+1, len(words))): # min and max used to avoid outOfBounds exeption
      if i != j:
        context_words.append(words[j])

    center_idx = word2idx[center_word]
    h = W[center_idx]

    for ctx_word in context_words:
      ctx_idx = word2idx[ctx_word]
      ctx_vector = W_prim[ctx_idx]

      negative_samples_index = np.random.choice(V, size=K, p=noise_Distribution)

      for k in range(K):
        while negative_samples_index[k] == ctx_idx:
              negative_samples_index[k] = np.random.choice(V, p=noise_Distribution)

      negative_samples_vectors = W_prim[negative_samples_index]

      pos_score = sigmoid_scalar(np.dot(h, ctx_vector))
      neg_score = sigmoid_vector(np.dot(negative_samples_vectors * -1, h))

      E = -1 * np.log(pos_score) - np.sum(np.log(neg_score)) # loss function
      total_loss+= E

      positive_error = pos_score - 1.0
      negative_error = 1.0 - neg_score

      EH = positive_error * ctx_vector + np.sum(negative_error[:, np.newaxis] * negative_samples_vectors, axis=0)

      W_prim[ctx_idx] -= learning_rate * positive_error * h

      W[center_idx] -= learning_rate * EH

      W_prim[negative_samples_index] -= learning_rate * np.outer(negative_error, h)

  print(f"Total loss for Epoch {epoch+1} is {total_loss}")

Total loss for Epoch 0 is 1395278.7047465325
Total loss for Epoch 1 is 1242853.9249304023
Total loss for Epoch 2 is 1194931.8114541566
Total loss for Epoch 3 is 1161611.4852748036
Total loss for Epoch 4 is 1130536.638968137
